# 04: Label Reliability Audit (Groq vs GPT-4o-mini)

Compares production labels from both classifiers on the same clean headline input.
Outputs go to `../data/audit/`.

**Run the full audit:** execute the cell below, or from terminal:
`python ../scripts/run_audit.py`

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

AUDIT = Path("../data/audit")
ROOT = Path("..").resolve()
EF02 = ROOT.parent / "EF-02"

# Run full audit (quality, export, agreement consolidation, gold eval, report)
subprocess.run([sys.executable, str(ROOT / "scripts" / "run_audit.py")], check=True)

CompletedProcess(args=['C:\\Users\\K2\\anaconda3\\python.exe', 'C:\\Users\\K2\\Documents\\DSAi\\Project\\EF-02 final\\EF-02-openai\\scripts\\run_audit.py'], returncode=0)

## Section 1: Data quality scorecard

In [2]:
metrics = json.loads((AUDIT / "quality_metrics.json").read_text(encoding="utf-8"))
pd.DataFrame([metrics["groq"], metrics["openai"]]).T

,0,1
name,Groq,OpenAI
total_rows,6235,6235
relevant_count,5163,4691
relevant_pct,82.8,75.2
failed_relevant_nan,0,0
relevant_null_sentiment,428,13
relevant_null_category,0,81
invalid_categories,0,0
sentiment_distribution,"{'Positive': 2168, 'Neutral': 2162, 'NaN': 428...","{'Positive': 3857, 'Neutral': 451, 'Negative':..."
positive_pct_of_relevant,42.0,82.2


## Section 2: Disagreement anatomy

In [3]:
print("Disagreement counts:")
for k, v in metrics["disagreement_counts"].items():
    print(f"  {k}: {v}")

examples = json.loads((AUDIT / "worked_examples.json").read_text(encoding="utf-8"))
for bucket, rows in examples.items():
    print(f"\n--- {bucket} (sample) ---")
    for r in rows:
        print(f"  [{r.get('sentiment_groq')} / {r.get('sentiment_oai')}] {r['headline'][:85]}")

Disagreement counts:
  groq_relevant_only: 644
  openai_relevant_only: 172
  neutral_to_positive: 1409
  negative_to_positive: 25
  groq_null_to_oai_positive: 264
  category_disagree: 1299
  full_agreement: 1925

--- groq_relevant_only (sample) ---
  [nan / nan] KIA sees surge in passengers amid increased flights
  [nan / nan] Boost for Akkuyu NPP as turbine installation completed
  [nan / nan] Govt highlights CRJE’s role in China-Tanzania ties
  [Neutral / nan] Airtel intensifies anti-fraud measures
  [Negative / nan] TBS destroys 18m/- substandard goods

--- neutral_to_positive (sample) ---
  [Neutral / Positive] Goods, services imports rise by 2.3 per cent
  [Neutral / Positive] Govt urges Rukwa diaspora to invest at home
  [Neutral / Positive] Tanzania seeks 7tri/- Chinese FDIs
  [Neutral / Positive] E-commerce set to thrive
  [Neutral / Positive] MOLLIS GROUP: A game changer in marketing African products in the continent

--- negative_to_positive (sample) ---
  [Negative / Positiv

## Section 3: Gold-set 3-in-1 evaluation

In [4]:
gold = json.loads((AUDIT / "gold_eval_scores.json").read_text(encoding="utf-8"))
pd.DataFrame(gold).T

,n,failed_batches,relevance_acc,sentiment_acc,category_acc,strict_3in1_acc,pred_relevant_pct,pred_positive_pct,pred_neutral_pct
groq,504.0,0.0,0.9901,0.8679,0.7862,0.6939,99.0,43.6,26.8
openai,504.0,0.0,0.9524,0.8954,0.8138,0.7280,95.2,57.7,14.2


## Section 4: Manual review

Open `../data/audit/disagreement_review_sample.csv`, fill `your_relevant`, `your_category`, `your_sentiment`, and `notes`, then re-run the cell below.

In [5]:
review = pd.read_csv(AUDIT / "disagreement_review_sample.csv")
filled = review[review["your_sentiment"].notna() & (review["your_sentiment"] != "")]
if len(filled) == 0:
    print("No manual labels yet — fill your_* columns in disagreement_review_sample.csv")
else:
    groq_match = (filled["your_sentiment"] == filled["sentiment_groq"]).mean() * 100
    oai_match = (filled["your_sentiment"] == filled["sentiment_openai"]).mean() * 100
    print(f"Human agreement with Groq sentiment: {groq_match:.1f}%")
    print(f"Human agreement with OpenAI sentiment: {oai_match:.1f}%")

No manual labels yet — fill your_* columns in disagreement_review_sample.csv


## Section 5: Correlation robustness (agreement-only)

In [ ]:
corr = json.loads((AUDIT / "correlation_comparison.json").read_text(encoding="utf-8"))
for label, data in corr.items():
    print(f"\n=== {label} ===")
    for pair, vals in data.items():
        sig = " *" if vals["significant"] else ""
        print(f"  {pair}: r={vals['r']}, p={vals['p']}{sig}")

## Section 6: Production recommendation

See `../data/audit/AUDIT_REPORT.md` for the full scored rubric and recommendation.

In [6]:
print((AUDIT / "AUDIT_REPORT.md").read_text(encoding="utf-8"))

# Label Reliability Audit Report

Groq (`llama-3.1-8b-instant`) vs OpenAI (`gpt-4o-mini`) on shared `tz_headlines_clean.csv` input.

## 1. Data quality scorecard

| Metric | Groq | OpenAI |
|--------|------|--------|
| Relevant rows | 5163 (82.8%) | 4691 (75.2%) |
| Relevant + null sentiment | **428** | 13 |
| Relevant + null category | 0 | **81** |
| Positive % of relevant | 42.0% | **82.2%** |
| Neutral % of relevant | 41.9% | 9.6% |
| Positive-bias alarm (>70%) | False | True |

**Groq anomaly:** 428 relevant headlines lack sentiment — likely incomplete retry after relabeling. These distort monthly sentiment aggregates until fixed.

**OpenAI anomaly:** 81 relevant rows lost category after schema enforcement; 82.2% positive rate vs Groq's 42.0% suggests systematic positive upgrading.

## 2. Disagreement anatomy

| Bucket | Count |
|--------|-------|
| Groq relevant only | 644 |
| OpenAI relevant only | 172 |
| Neutral → Positive | 1409 |
| Negative → Positive | 25 |
| Groq null → Ope